In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter

import pandas as pd
import matplotlib.pyplot as plt
import time
import os

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [14]:
import torchvision
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

path = r"C:\Users\KUNAL\Downloads\Dataset_DL\Dataset_DL"

train_dataset = torchvision.datasets.MNIST(
    root=path,
    train=True,
    download=False,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root=path,
    train=False,
    download=False,
    transform=transform
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 60000
Testing samples: 10000


In [15]:
class CNN(nn.Module):

    def __init__(self, layers=2):
        super().__init__()

        modules = []
        in_channels = 1

        for i in range(layers):

            out_channels = 32 if i == 0 else 64

            modules.append(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=3,
                    padding=1
                )
            )

            modules.append(nn.ReLU())
            modules.append(nn.MaxPool2d(2))

            in_channels = out_channels

        self.features = nn.Sequential(*modules)

        with torch.no_grad():
            dummy = torch.zeros(1, 1, 28, 28)
            output = self.features(dummy)
            feature_size = output.view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feature_size, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [16]:
experiments = [

    {
        "name": "exp_A",
        "lr": 0.001,
        "batch_size": 64,
        "layers": 2
    },

    {
        "name": "exp_B",
        "lr": 0.01,
        "batch_size": 64,
        "layers": 2
    },

    {
        "name": "exp_C",
        "lr": 0.0001,
        "batch_size": 64,
        "layers": 2
    },

    {
        "name": "exp_D",
        "lr": 0.001,
        "batch_size": 128,
        "layers": 2
    },

    {
        "name": "exp_E",
        "lr": 0.001,
        "batch_size": 32,
        "layers": 2
    },

    {
        "name": "exp_F",
        "lr": 0.001,
        "batch_size": 64,
        "layers": 3
    }
]

EPOCHS = 10

In [17]:
def train_experiment(config):

    print("\nStarting:", config["name"])

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config["batch_size"],
        shuffle=False
    )

    model = CNN(layers=config["layers"]).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=config["lr"]
    )

    writer = SummaryWriter(
        log_dir=os.path.join("runs", config["name"])
    )

    start_time = time.time()

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = total_loss / len(train_loader)
        train_accuracy = 100 * correct / total

        # Testing
        model.eval()

        test_loss_total = 0
        test_correct = 0
        test_total = 0

        with torch.no_grad():

            for images, labels in test_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                test_loss_total += loss.item()

                _, predicted = torch.max(outputs.data, 1)

                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        test_loss = test_loss_total / len(test_loader)
        test_accuracy = 100 * test_correct / test_total

        # TensorBoard logging
        writer.add_scalar("Loss/Train", train_loss, epoch + 1)
        writer.add_scalar("Loss/Test", test_loss, epoch + 1)

        writer.add_scalar("Accuracy/Train", train_accuracy, epoch + 1)
        writer.add_scalar("Accuracy/Test", test_accuracy, epoch + 1)

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} "
            f"Train Acc: {train_accuracy:.2f}% "
            f"Test Loss: {test_loss:.4f} "
            f"Test Acc: {test_accuracy:.2f}%"
        )

    training_time = time.time() - start_time

    writer.close()

    print("Training Time:", round(training_time, 2), "seconds")

    return {
        "Experiment": config["name"],
        "Learning Rate": config["lr"],
        "Batch Size": config["batch_size"],
        "Layers": config["layers"],
        "Train Loss": train_loss,
        "Train Accuracy": train_accuracy,
        "Test Loss": test_loss,
        "Test Accuracy": test_accuracy,
        "Training Time (sec)": training_time
    }

In [ ]:
results = []

for config in experiments:

    result = train_experiment(config)

    results.append(result)

In [ ]:
results_df = pd.DataFrame(results)

print("FINAL EXPERIMENT RESULTS")

display(results_df)

In [ ]:
results_sorted = results_df.sort_values(
    by="Test Accuracy",
    ascending=False
)

print("RESULTS SORTED BY TEST ACCURACY")

display(results_sorted)

In [ ]:
best = results_sorted.iloc[0]

print("BEST CONFIGURATION")

print("Experiment:", best["Experiment"])
print("Learning Rate:", best["Learning Rate"])
print("Batch Size:", best["Batch Size"])
print("Number of Layers:", best["Layers"])
print("Test Accuracy:", best["Test Accuracy"])
print("Test Loss:", best["Test Loss"])
print("Training Time:", best["Training Time (sec)"])

In [ ]:
results_df.to_csv(
    "mnist_hyperparameter_results.csv",
    index=False
)

print("Results saved successfully")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs